# Matching and Matched Inference: Singapore FY2023
Implements proposal Section 4(2)–(4): matching, sign test, permutation test, bootstrap CI and matched quantile gaps.

Load the cleaned data

In [25]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TAB_DIR = ROOT / "outputs" / "tables"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = ROOT / "data" / "processed" / "orbis_singapore_fy2023_clean.parquet"
df = pd.read_parquet(DATA_FILE)

print("File:", DATA_FILE.name)
print("Shape:", df.shape)
print(df.columns.tolist())

File: orbis_singapore_fy2023_clean.parquet
Shape: (48995, 32)
['Company name Latin alphabet', 'Inactive', 'Quoted', 'branch', 'OwnData', 'Woco', 'country_code', 'nace4', 'consolidation_code', 'bvd_id', 'incorporation_date', 'assets_th_usd', 'Total assets th USD 2022', 'revenue_th_usd', 'Operating revenue (Turnover) th USD 2022', 'employees', 'Number of employees 2022', 'ish_bvd_id', 'guo_bvd_id', 'ISH - Name', 'GUO - Name', 'SH - BvD Independence Indicator', 'listing_status', 'status', 'Closing date Last avail. yr', 'closing_date_2023', 'Closing date 2022', '_source_file', 'listed', 'assets_usd', 'log_assets', 'nace2']


Exclude subsidiaries of listed firms

Private firms whose immediate shareholder (ISH) or global ultimate owner (GUO) is a sampled listed firm are excluded, as specified in the proposal. Ownership links at intermediate levels are not captured, which is a limitation.

In [ ]:
# Exclude private subsidiaries of sampled listed firms (proposal Section 3)
listed_ids = set(eda.loc[eda["listing_status"] == "Listed", "bvd_id"].astype(str))
owner_is_listed = (eda["ish_bvd_id"].astype("string").isin(listed_ids).fillna(False)
                   | eda["guo_bvd_id"].astype("string").isin(listed_ids).fillna(False))
is_sub = (eda["listing_status"] == "Unlisted") & owner_is_listed

owners = pd.concat([eda.loc[is_sub, "ish_bvd_id"], eda.loc[is_sub, "guo_bvd_id"]]).dropna().astype(str)
print("Private firms whose ISH or GUO is a sampled listed firm:", int(is_sub.sum()))
print("Distinct listed parents involved:", owners[owners.isin(listed_ids)].nunique())

eda = eda[~is_sub].copy()
print("\nFirms by status after exclusion:\n", eda["listing_status"].value_counts())

Build the matching variables

In [26]:
# Listing status as text, from the cleaning script's 1/0 flag
df["listing_status"] = df["listed"].map({1: "Listed", 0: "Unlisted"})

# Incorporation year: the cleaned file lost year-only dates (e.g. "2006"),
# so take the year from the original text in the merged file, matched by firm ID
MERGED_FILE = ROOT / "data" / "interim" / "orbis_singapore_fy2023_merged.parquet"
raw = pd.read_parquet(MERGED_FILE, columns=["BvD ID number", "Date of incorporation"])
raw_year = pd.to_numeric(
    raw["Date of incorporation"].astype("string").str.extract(r"(\d{4})", expand=False),
    errors="coerce")
year_by_id = pd.Series(raw_year.to_numpy(), index=raw["BvD ID number"].astype("string"))
print("Duplicate IDs in merged file:", year_by_id.index.duplicated().sum())
year_by_id = year_by_id[~year_by_id.index.duplicated()]
df["incorporation_year"] = df["bvd_id"].astype("string").map(year_by_id)

# Actual age in FY2023
df["firm_age"] = 2023 - df["incorporation_year"]
df.loc[df["firm_age"] < 0, "firm_age"] = np.nan
df["age_band"] = pd.cut(df["firm_age"], bins=[-1, 4, 9, 19, float("inf")],
                        labels=["0–4", "5–9", "10–19", "20+"])

# Account type: C (consolidated) vs U (unconsolidated)
df["consolidated"] = df["consolidation_code"].astype("string").str.strip().str.upper().str[0]
df["consolidated"] = df["consolidated"].where(df["consolidated"].isin(["C", "U"]))

# Analysis sample: positive assets only (log_assets = ln(assets_usd), checked = 0 difference)
eda = df[df["listing_status"].notna() & (df["assets_usd"] > 0)].copy()

print("\nFirms by status:\n", eda["listing_status"].value_counts())
print("\nMissing matching variables:")
print(eda.groupby("listing_status")[["nace2", "firm_age", "age_band", "consolidated", "log_assets"]]
         .apply(lambda g: g.isna().sum()))

Duplicate IDs in merged file: 0

Firms by status:
 listing_status
Unlisted    47949
Listed        496
Name: count, dtype: int64

Missing matching variables:
                nace2  firm_age  age_band  consolidated  log_assets
listing_status                                                     
Listed              0         0         0             0           0
Unlisted            0         0         0             0           0


## Matching
For each listed firm, up to three private firms are selected without replacement from the same NACE 2-digit industry, age band and account type (C/U), choosing those closest in actual firm age. Ties are broken randomly (seed 607). Country and financial year are constant (Singapore, FY2023).

In [27]:
rng = np.random.default_rng(607)
KEYS = ["nace2", "age_band", "consolidated"]
ID = "bvd_id"
MAX_CONTROLS = 3

base = eda.dropna(subset=["log_assets", "firm_age"] + KEYS)
listed_df = base[base["listing_status"] == "Listed"]
pool      = base[base["listing_status"] == "Unlisted"]
pool_groups = {key: grp for key, grp in pool.groupby(KEYS, observed=True)}

sets, unmatched = [], []
n_sets = 0

for key, L in listed_df.groupby(KEYS, observed=True):
    L = L.sample(frac=1, random_state=int(rng.integers(1_000_000)))   # random order
    P = pool_groups.get(key)
    if P is None:
        unmatched += [{ID: i, "reason": "no private firm in same stratum"} for i in L[ID]]
        continue

    avail = P.assign(tiebreak=rng.random(len(P)))       # unused private firms
    picks = {idx: [] for idx in L.index}                 # listed firm -> its controls

    # Round-robin: everyone gets a 1st control, then a 2nd, then a 3rd
    for _ in range(MAX_CONTROLS):
        for idx, row in L.iterrows():
            if avail.empty:
                break
            gap = (avail["firm_age"] - row["firm_age"]).abs()
            best = avail.assign(age_gap=gap).sort_values(["age_gap", "tiebreak"]).index[0]
            picks[idx].append((best, gap[best]))
            avail = avail.drop(best)                      # no replacement

    for idx, row in L.iterrows():
        if not picks[idx]:
            unmatched.append({ID: row[ID], "reason": "stratum's private firms all used"})
            continue
        n_sets += 1
        sets.append({"set_id": n_sets, "role": "listed", ID: row[ID],
                     "firm_age": row["firm_age"], "age_gap": 0.0,
                     "log_assets": row["log_assets"], **{k: row[k] for k in KEYS}})
        for c_idx, c_gap in picks[idx]:
            c = P.loc[c_idx]
            sets.append({"set_id": n_sets, "role": "private", ID: c[ID],
                         "firm_age": c["firm_age"], "age_gap": c_gap,
                         "log_assets": c["log_assets"], **{k: c[k] for k in KEYS}})

matched_long = pd.DataFrame(sets)
n_ctrl = matched_long[matched_long["role"] == "private"].groupby("set_id").size()
matched_long["n_controls"] = matched_long["set_id"].map(n_ctrl)
unmatched = pd.DataFrame(unmatched)

print(f"Matched {matched_long['set_id'].nunique()} of {len(listed_df)} listed firms")
print("\nControls per set:\n", n_ctrl.value_counts().sort_index())
if len(unmatched):
    print("\nUnmatched reasons:\n", unmatched["reason"].value_counts())
print("\nAny private firm used twice?",
      matched_long.loc[matched_long["role"] == "private", ID].duplicated().any())

Matched 353 of 496 listed firms

Controls per set:
 1    112
2     51
3    190
Name: count, dtype: int64

Unmatched reasons:
 reason
no private firm in same stratum     89
stratum's private firms all used    54
Name: count, dtype: int64

Any private firm used twice? False


353 of 496 listed firms (71%) were matched. 89 had no private firm in the same industry, age band and account type, and 54 were in strata with more listed than private firms. Exact consolidation-code matching was infeasible (495 of 496 listed firms are C1, versus 1 private firm), so firms were matched on broad account type (C vs U).

## Matched gap and balance
D_i = ln(assets) of the listed firm − mean ln(assets) of its matched private firms.
Balance is checked on actual firm age before and after matching.

Dᵢ for each set

In [28]:
lst = matched_long[matched_long["role"] == "listed"].set_index("set_id")
prv = matched_long[matched_long["role"] == "private"]

sets_df = pd.DataFrame({
    "y_listed": lst["log_assets"],
    "y_private_mean": prv.groupby("set_id")["log_assets"].mean(),
    "n_controls": lst["n_controls"],
})
sets_df["D"] = sets_df["y_listed"] - sets_df["y_private_mean"]
sets_df["D"].describe().round(3)

count    353.000
mean       2.158
std        2.484
min       -7.945
25%        0.738
50%        2.014
75%        3.883
max       10.652
Name: D, dtype: float64

B: balance

In [29]:
def smd(a, b, wb=None):
    """Standardised mean difference; wb = weights for group b."""
    wb = np.ones(len(b)) if wb is None else np.asarray(wb)
    mb = np.average(b, weights=wb)
    vb = np.average((b - mb) ** 2, weights=wb)
    return (a.mean() - mb) / np.sqrt((a.var() + vb) / 2)

w_prv = 1 / prv["n_controls"]          # each matched set counts equally

balance = pd.DataFrame({
    "Listed mean age":  [listed_df["firm_age"].mean(), lst["firm_age"].mean()],
    "Private mean age": [pool["firm_age"].mean(),
                         np.average(prv["firm_age"], weights=w_prv)],
    "SMD (age)": [smd(listed_df["firm_age"], pool["firm_age"]),
                  smd(lst["firm_age"], prv["firm_age"], w_prv)],
}, index=["Before matching (all firms)", "After matching"]).round(3)
balance.to_csv(TAB_DIR / "matching_balance.csv")
print(balance)

print("\nAge gap to matched controls (years): median =", prv["age_gap"].median(),
      "| 90th pct =", prv["age_gap"].quantile(0.9), "| max =", prv["age_gap"].max())

# Matched vs unmatched listed firms
is_matched = listed_df[ID].isin(lst[ID])
print("\nMatched vs unmatched listed firms:")
print(listed_df.assign(matched=is_matched).groupby("matched")
      .agg(n=(ID, "size"), mean_age=("firm_age", "mean"),
           median_log_assets=("log_assets", "median")).round(2))

# Match rate by age band and top industries
rate = lambda col: (listed_df.assign(matched=is_matched)
                    .groupby(col, observed=True)["matched"]
                    .agg(n_listed="size", match_rate="mean").round(2))
print("\nMatch rate by age band:\n", rate("age_band"))
print("\nMatch rate, 10 largest industries:\n",
      rate("nace2").sort_values("n_listed", ascending=False).head(10))

                             Listed mean age  Private mean age  SMD (age)
Before matching (all firms)           24.591            11.563      0.851
After matching                        23.742            21.919      0.110

Age gap to matched controls (years): median = 1.0 | 90th pct = 10.0 | max = 90.0

Matched vs unmatched listed firms:
           n  mean_age  median_log_assets
matched                                  
False    143     26.69              18.70
True     353     23.74              18.32

Match rate by age band:
           n_listed  match_rate
age_band                      
0–4             26        0.96
5–9             62        0.85
10–19          164        0.63
20+            244        0.70

Match rate, 10 largest industries:
        n_listed  match_rate
nace2                      
68           38        1.00
26           38        0.47
46           31        1.00
94           27        0.04
10           20        1.00
28           19        1.00
47           18    

Matching substantially improved age balance: the standardised mean difference in firm age fell from 0.85 to 0.11. The median age gap between a listed firm and its controls is 1 year (90th percentile 10 years); larger gaps occur within the open-ended 20+ band. Unmatched listed firms are slightly older and larger than matched ones, so matched results describe the 353 matched firms. Electronics (NACE 26) has the lowest match rate (47%).

## Matched inference
Following proposal Section 4(3): one-sided exact sign test, within-set permutation test of the median gap (9,999 permutations), and 95% percentile bootstrap CI for the median gap (5,000 resamples of whole matched sets).

one-sided sign test

In [30]:
D = sets_df["D"].to_numpy()
nz = D[D != 0]
k = int((nz > 0).sum())
sign = stats.binomtest(k, len(nz), p=0.5, alternative="greater")
print(f"H0: P(D > 0) = 0.5   vs   H1: P(D > 0) > 0.5")
print(f"Listed larger in {k}/{len(nz)} sets ({k/len(nz):.1%}), one-sided p = {sign.pvalue:.3g}")

H0: P(D > 0) = 0.5   vs   H1: P(D > 0) > 0.5
Listed larger in 305/353 sets (86.4%), one-sided p = 3.59e-47


The listed firm is larger than its matched private firms in 305 of 353 sets (86.4%; one-sided sign test p < 0.001). The median matched gap is 2.01 on the natural-log scale, i.e. the median listed firm is about 7.5 times larger than comparable private firms (within-set permutation test, one-sided p < 0.001; 95% bootstrap CI 5.9× to 10.5×). These results are associations, not causal effects of listing.

within-set permutation test of the median gap

In [31]:
obs_median = np.median(D)
N_PERM = 9_999
perm_D = np.empty((N_PERM, len(sets_df)))
col = 0
for m, grp in matched_long.groupby("n_controls"):
    Y = (grp.sort_values(["set_id", "role"])            # listed first within each set
            .groupby("set_id")["log_assets"].apply(np.array))
    Y = np.vstack(Y.to_numpy())                         # rows = sets, column 0 = listed
    total = Y.sum(axis=1)
    j = rng.integers(0, m + 1, size=(N_PERM, len(Y)))   # who gets the "listed" label
    yj = Y[np.arange(len(Y)), j]
    perm_D[:, col:col + len(Y)] = yj - (total - yj) / m
    col += len(Y)

perm_medians = np.median(perm_D, axis=1)
p_perm = (1 + np.sum(perm_medians >= obs_median)) / (N_PERM + 1)
print(f"Observed median gap = {obs_median:.3f} (≈ {np.exp(obs_median):.1f}× larger)")
print(f"One-sided permutation p = {p_perm:.4f}")

Observed median gap = 2.014 (≈ 7.5× larger)
One-sided permutation p = 0.0001


bootstrap CI for the median gap

In [32]:
B = 5_000
idx = rng.integers(0, len(D), size=(B, len(D)))      # resample whole matched sets
boot_medians = np.median(D[idx], axis=1)
lo, hi = np.percentile(boot_medians, [2.5, 97.5])
print(f"95% CI for median D: [{lo:.3f}, {hi:.3f}]")
print(f"→ the median listed firm is {np.exp(lo):.1f}× to {np.exp(hi):.1f}× larger than its matched private firms")

95% CI for median D: [1.774, 2.353]
→ the median listed firm is 5.9× to 10.5× larger than its matched private firms


## Matched quantile gaps
Δ_q = Q_listed(q) − Q_private(q) at q = 0.25, 0.50, 0.75, 0.90, using matched firms only, with equal total weight per matched set (each private control weighted 1/n_controls). 95% CIs from 5,000 matched-set bootstrap resamples.

In [33]:
def wquantile(x, w, q):
    """Weighted quantile (inverse-CDF definition)."""
    order = np.argsort(x)
    x, w = x[order], w[order]
    cw = np.cumsum(w) / w.sum()
    return x[np.searchsorted(cw, q)]

set_ids = sets_df.index.to_numpy()
yl = sets_df["y_listed"].to_numpy()
prv_by_set = {s: g["log_assets"].to_numpy() for s, g in prv.groupby("set_id")}

def gaps(pos, qs):
    L = yl[pos]
    P = np.concatenate([prv_by_set[set_ids[p]] for p in pos])
    W = np.concatenate([np.full(len(prv_by_set[set_ids[p]]),
                                1 / len(prv_by_set[set_ids[p]])) for p in pos])
    return np.array([np.quantile(L, q, method="inverted_cdf") - wquantile(P, W, q)
                     for q in qs])

QS = [0.25, 0.50, 0.75, 0.90]
point = gaps(np.arange(len(set_ids)), QS)
boot = np.array([gaps(rng.integers(0, len(set_ids), len(set_ids)), QS)
                 for _ in range(5_000)])
ci = np.percentile(boot, [2.5, 97.5], axis=0)

matched_qgaps = pd.DataFrame({
    "quantile": [f"Q{int(q*100)}" for q in QS],
    "ln_gap": point, "ci_low": ci[0], "ci_high": ci[1],
    "ratio": np.exp(point), "ratio_low": np.exp(ci[0]), "ratio_high": np.exp(ci[1]),
}).round(3)
matched_qgaps.to_csv(TAB_DIR / "matched_quantile_gaps.csv", index=False)
matched_qgaps

,quantile,ln_gap,ci_low,ci_high,ratio,ratio_low,ratio_high
0,Q25,1.995,1.812,2.379,7.353,6.124,10.792
1,Q50,2.000,1.637,2.286,7.388,5.140,9.840
2,Q75,2.279,1.829,2.548,9.768,6.227,12.779
3,Q90,1.990,1.519,2.525,7.313,4.570,12.486


After matching, the size gap is similar across the distribution (about 7–10× at Q25, Q50, Q75 and Q90), with heavily overlapping confidence intervals. This indicates a broad upward shift rather than an effect concentrated in the upper tail. Compared with the unadjusted gaps (roughly 70–200×), matching on industry, age and account type explains most of the raw difference, but a substantial gap remains.

In [34]:
summary = pd.DataFrame({
    "statistic": ["Listed firms", "Matched sets", "Unmatched: no private firm in stratum",
                  "Unmatched: stratum exhausted", "Age SMD before", "Age SMD after",
                  "Sign test: share listed larger", "Sign test one-sided p",
                  "Median D (ln)", "Median ratio", "Permutation one-sided p",
                  "Bootstrap 95% CI low (ratio)", "Bootstrap 95% CI high (ratio)"],
    "value": [len(listed_df), len(sets_df),
              (unmatched["reason"] == "no private firm in same stratum").sum(),
              (unmatched["reason"] == "stratum's private firms all used").sum(),
              balance.iloc[0]["SMD (age)"], balance.iloc[1]["SMD (age)"],
              round(k / len(nz), 3), f"{sign.pvalue:.3g}",
              round(obs_median, 3), round(np.exp(obs_median), 2), f"{p_perm:.4f}",
              round(np.exp(lo), 2), round(np.exp(hi), 2)],
})
summary.to_csv(TAB_DIR / "matched_inference_summary.csv", index=False)
summary

,statistic,value
0,Listed firms,496
1,Matched sets,353
2,Unmatched: no private firm in stratum,89
3,Unmatched: stratum exhausted,54
4,Age SMD before,0.851
5,Age SMD after,0.11
6,Sign test: share listed larger,0.864
7,Sign test one-sided p,3.59e-47
8,Median D (ln),2.014
9,Median ratio,7.49
